# Inverse CDF Sampling, Numerical Inversion, and a Bridge to GANs

This notebook illustrates how to simulate data from a target distribution using the **inverse transform method**.

It includes:

- the basic inverse-CDF idea,
- **numerical inversion** when no closed form is available,
- **empirical CDF vs theoretical CDF** convergence,
- an **animated transformation** from uniform samples to target-distribution samples,
- and a **bridge to GANs**, where the inverse CDF is replaced by a small neural network.

This is a natural transition toward generative models: instead of knowing the exact inverse CDF, we can **learn a transport map** from a simple noise distribution to the target distribution.

The GAN viewpoint itself is closely aligned with the idea of starting from a simple latent distribution and learning a mapping toward the data distribution, as emphasized in the original GAN paper and later mathematical expositions.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import stats
from scipy.interpolate import interp1d
from scipy.optimize import brentq

import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams["figure.figsize"] = (8, 4.5)

ModuleNotFoundError: No module named 'scipy'

## 1. Mathematical idea

Let $F$ be the CDF of a target random variable $X$.  
If $U \sim \mathcal U(0,1)$, then

$$
X = F^{-1}(U)
$$

has distribution $F$.

Indeed, for every $x$,

$$
\mathbb P(F^{-1}(U)\le x)
=
\mathbb P(U \le F(x))
=
F(x),
$$

since $U$ is uniform on $[0,1]$.

So simulation reduces to two ingredients:

1. sample $U \sim \mathcal U(0,1)$,
2. apply the inverse CDF.

When $F^{-1}$ is explicit, this is direct.  
When it is not, we can still do **numerical inversion**.

## 2. Warm-up: exponential distribution with closed-form inverse

For an exponential distribution with rate $\lambda > 0$,

$$
F(x)=1-e^{-\lambda x}, \qquad x\ge 0.
$$

Solving $u = 1-e^{-\lambda x}$ gives

$$
F^{-1}(u)= -\frac{1}{\lambda}\log(1-u).
$$

Because \(1-U\) is again uniform on \([0,1]\), one often writes

$$
F^{-1}(U)= -\frac{1}{\lambda}\log U.
$$

In [ ]:
def inverse_cdf_exponential(u, lam=1.0):
    u = np.asarray(u)
    return -np.log(1 - u) / lam

n = 10_000
lam = 1.5
u = np.random.rand(n)
x = inverse_cdf_exponential(u, lam=lam)

grid = np.linspace(0, np.percentile(x, 99.5), 400)
pdf = lam * np.exp(-lam * grid)

plt.hist(x, bins=60, density=True, alpha=0.6, label="Simulated samples")
plt.plot(grid, pdf, lw=2, label="Theoretical PDF")
plt.title("Exponential distribution from inverse transform sampling")
plt.xlabel("x")
plt.ylabel("density")
plt.legend()
plt.show()

## 3. Empirical CDF vs theoretical CDF

A good way to see convergence is to compare

- the **empirical CDF**
$$
\widehat F_n(x)=\frac1n\sum_{i=1}^n \mathbf 1_{X_i\le x},
$$
- with the **true CDF** $F(x)$.

As $n$ grows, $\widehat F_n$ converges to $F$.

In [ ]:
def empirical_cdf(samples):
    xs = np.sort(samples)
    ys = np.arange(1, len(xs)+1) / len(xs)
    return xs, ys

sample_sizes = [50, 200, 1000, 5000]
lam = 1.5

grid = np.linspace(0, 4, 500)
true_cdf = 1 - np.exp(-lam * grid)

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, n in zip(axes, sample_sizes):
    u = np.random.rand(n)
    x = inverse_cdf_exponential(u, lam=lam)
    xs, ys = empirical_cdf(x)
    ax.step(xs, ys, where="post", label=f"Empirical CDF (n={n})")
    ax.plot(grid, true_cdf, lw=2, label="Theoretical CDF")
    ax.set_title(f"n = {n}")
    ax.grid(alpha=0.25)

axes[0].legend(loc="lower right")
for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("CDF")

fig.suptitle("Empirical CDF converges to the theoretical CDF", y=1.02)
fig.tight_layout()
plt.show()

We can also quantify the discrepancy using the Kolmogorov-type sup norm:

$$
\sup_x |\widehat F_n(x)-F(x)|.
$$

This gives a simple numerical measure of convergence.

In [ ]:
def ks_sup_distance(samples, cdf):
    xs = np.sort(samples)
    n = len(xs)
    emp = np.arange(1, n+1) / n
    return np.max(np.abs(emp - cdf(xs)))

sample_sizes = np.array([20, 50, 100, 200, 500, 1000, 2000, 5000])
distances = []

for n in sample_sizes:
    rep = []
    for _ in range(20):
        u = np.random.rand(n)
        x = inverse_cdf_exponential(u, lam=lam)
        rep.append(ks_sup_distance(x, lambda t: 1 - np.exp(-lam * t)))
    distances.append(np.mean(rep))

plt.plot(sample_sizes, distances, marker="o")
plt.xscale("log")
plt.yscale("log")
plt.title("Average sup-norm discrepancy between empirical and theoretical CDF")
plt.xlabel("Sample size n (log scale)")
plt.ylabel("Mean sup distance (log scale)")
plt.grid(alpha=0.3)
plt.show()

## 4. Numerical inversion when no closed form is available

Sometimes the CDF $F$ is known, but its inverse $F^{-1}$ has **no simple closed form**.

There are two common strategies:

1. **root finding**: for each $u \in (0,1)$, solve $F(x)-u=0$,
2. **grid/interpolation**: tabulate the CDF on a grid, then numerically invert it.

We illustrate both on a nontrivial example.

### Example: standard normal distribution

Its CDF has no elementary closed form.  
Of course, libraries provide `scipy.stats.norm.ppf`, but it is useful to build the inverse numerically.

In [ ]:
normal_cdf = stats.norm.cdf

def inverse_cdf_root(u, cdf, a=-10, b=10):
    '''
    Numerically invert a monotone CDF using root finding.
    Solves cdf(x) = u on [a,b].
    '''
    u = np.asarray(u)
    out = np.empty_like(u, dtype=float)
    for i, ui in enumerate(u):
        f = lambda x: cdf(x) - ui
        out[i] = brentq(f, a, b)
    return out

u_demo = np.array([0.01, 0.1, 0.5, 0.9, 0.99])
x_num = inverse_cdf_root(u_demo, normal_cdf)
x_ref = stats.norm.ppf(u_demo)

print("u       =", u_demo)
print("numeric =", np.round(x_num, 6))
print("scipy   =", np.round(x_ref, 6))
print("max abs difference =", np.max(np.abs(x_num - x_ref)))

In [ ]:
def build_inverse_cdf_interpolator(cdf, x_min=-8, x_max=8, num=20_000):
    x_grid = np.linspace(x_min, x_max, num)
    u_grid = cdf(x_grid)
    keep = np.concatenate([[True], np.diff(u_grid) > 0])
    u_grid = u_grid[keep]
    x_grid = x_grid[keep]
    return interp1d(
        u_grid,
        x_grid,
        kind="linear",
        bounds_error=False,
        fill_value=(x_grid[0], x_grid[-1]),
    )

normal_inv_interp = build_inverse_cdf_interpolator(stats.norm.cdf)

u_demo = np.linspace(0.01, 0.99, 7)
x_interp = normal_inv_interp(u_demo)
x_ref = stats.norm.ppf(u_demo)

print("u:", np.round(u_demo, 3))
print("interp :", np.round(x_interp, 6))
print("scipy  :", np.round(x_ref, 6))
print("max abs difference =", np.max(np.abs(x_interp - x_ref)))

### Sampling from the Gaussian by numerical inversion

In [ ]:
n = 5000
u = np.random.rand(n)
x_root = inverse_cdf_root(u, stats.norm.cdf, a=-8, b=8)
x_interp = normal_inv_interp(u)

grid = np.linspace(-4, 4, 400)
pdf = stats.norm.pdf(grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].hist(x_root, bins=60, density=True, alpha=0.6, label="Numerical root inversion")
axes[0].plot(grid, pdf, lw=2, label="Theoretical PDF")
axes[0].set_title("Gaussian via root-finding inversion")
axes[0].legend()

axes[1].hist(x_interp, bins=60, density=True, alpha=0.6, label="Interpolation inversion")
axes[1].plot(grid, pdf, lw=2, label="Theoretical PDF")
axes[1].set_title("Gaussian via interpolation inversion")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("density")

fig.tight_layout()
plt.show()

In [ ]:
u_test = np.linspace(0.001, 0.999, 1000)
x_exact = stats.norm.ppf(u_test)
x_root = inverse_cdf_root(u_test, stats.norm.cdf, a=-8, b=8)
x_interp = normal_inv_interp(u_test)

print("Root inversion max abs error :", np.max(np.abs(x_root - x_exact)))
print("Interpolation max abs error  :", np.max(np.abs(x_interp - x_exact)))

## 5. Animated transformation: from uniform samples to target samples

A very intuitive picture is the following:

- start with points \(u \in [0,1]\),
- gradually move them along a path from \(u\) to \(F^{-1}(u)\).

This is not the only possible transport path, but it is visually helpful.

Below, we animate the transformation for the exponential target.

In [ ]:
n_anim = 300
u_anim = np.random.rand(n_anim)
x_target = inverse_cdf_exponential(u_anim, lam=1.2)

frames = 60
t_values = np.linspace(0, 1, frames)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.set_xlim(-0.1, max(4, np.percentile(x_target, 99)))
ax.set_ylim(0, 1.5)
ax.set_title("Animated transformation: Uniform samples → Exponential samples")
ax.set_xlabel("value")
ax.set_yticks([])

scatter = ax.scatter(u_anim, np.zeros_like(u_anim) + 0.8, s=20, alpha=0.7)

grid = np.linspace(0, max(4, np.percentile(x_target, 99)), 400)
pdf = 1.2 * np.exp(-1.2 * grid)
ax2 = ax.twinx()
ax2.plot(grid, pdf, lw=2)
ax2.set_ylabel("target density")
ax2.set_ylim(0, pdf.max() * 1.15)

txt = ax.text(0.02, 0.92, "", transform=ax.transAxes)

def update(frame):
    t = t_values[frame]
    pos = (1 - t) * u_anim + t * x_target
    scatter.set_offsets(np.c_[pos, np.zeros_like(pos) + 0.8])
    txt.set_text(f"interpolation t = {t:.2f}")
    return scatter, txt

anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
plt.close(fig)

HTML(anim.to_jshtml())

If you want to export the animation as a GIF or MP4 later, you can use `anim.save(...)` provided the relevant writer is installed.

## 6. A second example with numerical inversion only: Beta distribution

To reinforce the idea, let us take a distribution on $[0,1]$ with a nontrivial shape, for example $\mathrm{Beta}(2,5)$.

Its CDF is available numerically, but we will invert it ourselves by interpolation.

In [ ]:
beta_dist = stats.beta(a=2, b=5)
beta_cdf = beta_dist.cdf
beta_pdf = beta_dist.pdf

beta_inv_interp = build_inverse_cdf_interpolator(beta_cdf, x_min=0, x_max=1, num=20_000)

u = np.random.rand(8000)
x_beta = beta_inv_interp(u)

grid = np.linspace(0, 1, 500)

plt.hist(x_beta, bins=60, density=True, alpha=0.6, label="Samples by numerical inversion")
plt.plot(grid, beta_pdf(grid), lw=2, label="Theoretical Beta(2,5) PDF")
plt.title("Beta distribution via numerical inverse CDF")
plt.xlabel("x")
plt.ylabel("density")
plt.legend()
plt.show()

In [ ]:
sample_sizes = [50, 200, 1000, 5000]
grid = np.linspace(0, 1, 500)
true_cdf = beta_cdf(grid)

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, n in zip(axes, sample_sizes):
    u = np.random.rand(n)
    x = beta_inv_interp(u)
    xs, ys = empirical_cdf(x)
    ax.step(xs, ys, where="post", label=f"Empirical CDF (n={n})")
    ax.plot(grid, true_cdf, lw=2, label="Theoretical CDF")
    ax.set_title(f"Beta(2,5), n={n}")
    ax.grid(alpha=0.25)

axes[0].legend(loc="lower right")
for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("CDF")

fig.suptitle("Empirical vs theoretical CDF for a numerically inverted distribution", y=1.02)
fig.tight_layout()
plt.show()

## 7. Bridge to GANs: replacing the inverse CDF with a neural network

The inverse-CDF method says:

$$
z \sim \mathcal U(0,1), \qquad x = F^{-1}(z).
$$

This already has the structure of a generative model:

- **latent variable**: $z$,
- **generator map**: $G(z)=F^{-1}(z)$.

In GANs, we typically do **not** know the exact inverse CDF or transport map.  
Instead, we learn a parameterized map $G_\theta$ from a simple latent distribution to the target data distribution. This is one of the central ideas of GANs. fileciteturn0file1

For a one-dimensional target, a neural network can be viewed as a learned approximation of the inverse-CDF transport. This is not yet a full GAN training loop, but it gives the right intuition:
- **inverse transform sampling** = exact map known,
- **learned generator** = exact map unknown, approximated from data.

The mathematical formulation of GANs precisely builds on this idea of learning a map from a simple source distribution toward the target distribution. fileciteturn0file0 fileciteturn0file2

### A simple neural-network approximation of the inverse CDF

We will train a small neural network $G_\theta$ on pairs

$$
(u_i, F^{-1}(u_i))
$$

for the target $\mathrm{Beta}(2,5)$, and then compare:

- exact inverse-CDF samples,
- neural-network-generated samples.

This is a **bridge** to GANs, not a GAN itself: here we supervise the map directly.

In [ ]:
n_train = 5000
u_train = np.random.rand(n_train, 1).astype(np.float32)
x_train = beta_inv_interp(u_train.squeeze()).reshape(-1, 1).astype(np.float32)

u_t = torch.from_numpy(u_train)
x_t = torch.from_numpy(x_train)

class Generator1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, z):
        return self.net(z)

gen = Generator1D()
optimizer = optim.Adam(gen.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

loss_history = []
epochs = 800

for epoch in range(epochs):
    optimizer.zero_grad()
    pred = gen(u_t)
    loss = loss_fn(pred, x_t)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

plt.plot(loss_history)
plt.title("Training loss for neural approximation of the inverse CDF")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
u_plot = np.linspace(0.001, 0.999, 500).reshape(-1, 1).astype(np.float32)
with torch.no_grad():
    x_nn = gen(torch.from_numpy(u_plot)).numpy().squeeze()

x_true = beta_inv_interp(u_plot.squeeze())

plt.plot(u_plot.squeeze(), x_true, lw=2, label="True numerical inverse CDF")
plt.plot(u_plot.squeeze(), x_nn, lw=2, label="Neural approximation")
plt.title("Inverse CDF vs neural network approximation")
plt.xlabel("u")
plt.ylabel("x")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
n_gen = 8000
z = np.random.rand(n_gen, 1).astype(np.float32)
with torch.no_grad():
    x_gen = gen(torch.from_numpy(z)).numpy().squeeze()

x_exact = beta_inv_interp(z.squeeze())

grid = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].hist(x_exact, bins=60, density=True, alpha=0.6, label="Exact inverse-CDF samples")
axes[0].plot(grid, beta_pdf(grid), lw=2, label="Target PDF")
axes[0].set_title("Exact inverse-CDF sampling")
axes[0].legend()

axes[1].hist(x_gen, bins=60, density=True, alpha=0.6, label="Neural generator samples")
axes[1].plot(grid, beta_pdf(grid), lw=2, label="Target PDF")
axes[1].set_title("Neural approximation of the transport map")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("density")

fig.tight_layout()
plt.show()

### What this teaches us

In 1D, inverse transform sampling gives an exact generator:

$$
G(u)=F^{-1}(u).
$$

A neural network can approximate this map.  
In higher dimension, there is generally **no single scalar CDF inversion** available, and the problem becomes much harder. This is where generative models such as GANs become important: they learn a mapping from a simple latent distribution to the data distribution without requiring an explicit inverse CDF. 

In other words:

- **inverse CDF method**: exact transport known,
- **neural transport**: transport approximated,
- **GAN**: transport learned from data through an adversarial objective.

## 8. Suggested exercises

1. Replace the Beta target by another distribution with bounded support.
2. Compare **root finding** and **interpolation** in terms of speed and error.
3. Estimate the convergence rate of the empirical CDF numerically.
4. Replace the supervised neural fit by a simple distribution-matching loss on samples.
5. For a 1D Gaussian target, train a neural network to approximate the quantile function and compare it with `norm.ppf`.

## 9. Final takeaway

This notebook shows a clean conceptual progression:

1. start from **uniform noise**,
2. transform it using an **inverse CDF**,
3. use **numerical inversion** when the inverse is not explicit,
4. observe **empirical convergence** of the simulated law,
5. animate the **transport from latent noise to target samples**,
6. then replace the exact inverse map by a **neural network**.

That last step is one of the most useful pedagogical bridges toward GANs and modern generative modeling.